### 문항 1 네이버 VIBE Top 100 수집

In [ ]:
# 대상: https://vibe.naver.com/chart

# 추출 필드: 순위 / 곡명 / 아티스트

# 아티스트는 리스트로 담을 것 (협업곡은 여러 명)

# Selenium 사용 금지

# 100건이 모두 수집되었는지 건수로 확인할 것

# 결과를 vibe_top100.csv로 저장할 것

In [6]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

In [10]:
URL = 'https://vibe.naver.com/chart'

API = 'https://apis.naver.com/vibeWeb/musicapiweb/vibe/v1/chart/track/total'

PARAMS = {'start': 1, 'display': 100}

HEADERS = {'User-Agent':'Mozilla 5.0','Accept':'application/json'}

res = requests.get(API, params={**PARAMS}, headers=HEADERS)

res.status_code


200

In [ ]:
res.json()['response']['result']['chart']['items']['tracks']

In [ ]:
items = res.json()['response']['result']['chart']['items']['tracks']
chart = []
for item in items:
    chart.append({
        '순위': item['rank']['currentRank'],
        '곡명': item['trackTitle'],
        '아티스트': [p.get('artistName', '') for p in item['artists']],
    })
chart

In [ ]:
## 최적화
URL = 'https://vibe.naver.com/chart'

API = 'https://apis.naver.com/vibeWeb/musicapiweb/vibe/v1/chart/track/total'

PARAMS = {'start': 1, 'display': 100}

HEADERS = {'User-Agent':'Mozilla 5.0','Accept':'application/json'}

def fetch_vibe_chart():
    res = requests.get(API, params={**PARAMS}, headers=HEADERS)
    res.raise_for_status
    items = res.json()['response']['result']['chart']['items']['tracks']
    chart = []
    for item in items:
        chart.append({
            '순위': item['rank']['currentRank'],
            '곡명': item['trackTitle'],
            '아티스트': [p.get('artistName', '') for p in item['artists']],
        })
    return chart

chart = fetch_vibe_chart()
print(f'{len(chart)}건 수집 완료')
df = pd.DataFrame(chart)
df.to_csv('vibe_top100.csv', index=False, encoding='utf-8-sig')


100건 수집 완료


### 문항 2 삼성전자 일별 시세 1년치 수집

In [ ]:
# 대상: https://finance.naver.com/item/sise.naver?code=005930

# 추출 필드: 날짜 / 종가 / 전일비 / 시가 / 고가 / 저가 / 거래량

# 1년치 전량을 페이지네이션으로 수집할 것

# 숫자는 정제하여 숫자로 변환할 것

# 마지막 페이지를 자동으로 감지해 종료할 것

# 결과를 samsung_1y.csv로 저장할 것

In [13]:
URL = 'https://finance.naver.com/item/sise_day.naver'

HEADERS = {'User-Agent':'Mozilla 5.0'}

res = requests.get(URL, params={'code':'005930','page':1}, headers=HEADERS)
res.status_code

200

In [16]:
soup = BeautifulSoup(res.text, 'html.parser')
row = []
trs = soup.select('table.type2 tr')
for tr in trs:
    td = tr.select('td')
    if len(td) < 7 :
        continue
    plus_mius = td[2].select_one('em.bu_p').text.strip()
    amount = int((td[2].select_one('span.tah').text.strip()).replace(',',''))
    row.append({
        '날짜': td[0].text.strip(),
        '종가': int(td[1].text.strip().replace(',','')),
        '전일비':f'{plus_mius} {amount}',
        '시가': int(td[3].text.strip().replace(',','')),
        '고가':int(td[4].text.strip().replace(',','')),
        '저가':int(td[5].text.strip().replace(',','')),
        '거래량':int(td[6].text.strip().replace(',','')),
    })
row



[{'날짜': '2026.08.24',
  '종가': 258500,
  '전일비': '하락 23000',
  '시가': 271500,
  '고가': 272000,
  '저가': 255000,
  '거래량': 26759336},
 {'날짜': '2026.08.21',
  '종가': 281500,
  '전일비': '상승 10500',
  '시가': 267000,
  '고가': 285000,
  '저가': 266000,
  '거래량': 27746471},
 {'날짜': '2026.08.20',
  '종가': 271000,
  '전일비': '상승 23500',
  '시가': 257000,
  '고가': 273000,
  '저가': 252500,
  '거래량': 26095919},
 {'날짜': '2026.08.19',
  '종가': 247500,
  '전일비': '하락 21000',
  '시가': 251500,
  '고가': 254500,
  '저가': 246500,
  '거래량': 22788552},
 {'날짜': '2026.08.18',
  '종가': 268500,
  '전일비': '하락 6000',
  '시가': 283000,
  '고가': 288000,
  '저가': 265000,
  '거래량': 24464621},
 {'날짜': '2026.08.14',
  '종가': 274500,
  '전일비': '상승 6500',
  '시가': 275000,
  '고가': 275500,
  '저가': 266000,
  '거래량': 21669476},
 {'날짜': '2026.08.13',
  '종가': 268000,
  '전일비': '상승 12500',
  '시가': 267500,
  '고가': 271000,
  '저가': 262500,
  '거래량': 35530867},
 {'날짜': '2026.08.12',
  '종가': 255500,
  '전일비': '상승 16000',
  '시가': 243500,
  '고가': 260500,
  '저가': 241000,
  '거래량

In [34]:
## 최적화 (숫자 변환 함수도 필요)
import re 
from datetime import datetime, timedelta

URL = 'https://finance.naver.com/item/sise_day.naver'

HEADERS = {'User-Agent':'Mozilla 5.0'}

def get_num(a):
    m = re.search(r'\d+', a.replace(',','').strip())
    return int(m.group()) if m else ''

def fetch(page):
    res = requests.get(URL, params={'code':'005930','page':page}, headers=HEADERS)
    res.raise_for_status
    return res.text

def parse(html):
    soup = BeautifulSoup(html, 'html.parser')
    row = []
    trs = soup.select('table.type2 tr')
    for tr in trs:
        td = tr.select('td')
        if len(td) < 7 :
            continue
        plus_mius = td[2].select_one('em.bu_p').text.strip()
        amount = get_num(td[2].select_one('span.tah').text)
        row.append({
            '날짜': td[0].text.strip(),
            '종가': get_num(td[1].text),
            '전일비':f'{plus_mius} {amount}',
            '시가': get_num(td[3].text),
            '고가':get_num(td[4].text),
            '저가':get_num(td[5].text),
            '거래량':get_num(td[6].text),
        })
    return row

result = []

for page in range(1, 100):
    rows = parse(fetch(page))

    if not rows:
        print(f'{page}페이지가 비어 있습니다. 크롤링 종료')
        break

    last_date = rows[0]['날짜']

    if datetime.strptime(last_date, '%Y.%m.%d') < datetime.now() - timedelta(365):
        print(f'{last_date}, 1년 이전 날짜까지 데이터크롤링 완료')
        break
    result.extend(rows)
    print(f'{page}페이지, 누적{len(result)}건 수집 최종{rows[-1]['날짜']} ')
    time.sleep(0.7)


df = pd.DataFrame(result).drop_duplicates(subset=["날짜"])
df.to_csv('samsung_1y.csv', index=False, encoding='utf-8-sig')
print(f'최종 {len(df)}건 · {df['날짜'].min()} ~ {df['날짜'].max()}')


1페이지, 누적10건 수집 최종2026.08.10 
2페이지, 누적20건 수집 최종2026.07.27 
3페이지, 누적30건 수집 최종2026.07.10 
4페이지, 누적40건 수집 최종2026.06.26 
5페이지, 누적50건 수집 최종2026.06.12 
6페이지, 누적60건 수집 최종2026.05.28 
7페이지, 누적70건 수집 최종2026.05.13 
8페이지, 누적80건 수집 최종2026.04.27 
9페이지, 누적90건 수집 최종2026.04.13 
10페이지, 누적100건 수집 최종2026.03.30 
11페이지, 누적110건 수집 최종2026.03.16 
12페이지, 누적120건 수집 최종2026.02.27 
13페이지, 누적130건 수집 최종2026.02.10 
14페이지, 누적140건 수집 최종2026.01.27 
15페이지, 누적150건 수집 최종2026.01.13 
16페이지, 누적160건 수집 최종2025.12.26 
17페이지, 누적170건 수집 최종2025.12.11 
18페이지, 누적180건 수집 최종2025.11.27 
19페이지, 누적190건 수집 최종2025.11.13 
20페이지, 누적200건 수집 최종2025.10.30 
21페이지, 누적210건 수집 최종2025.10.16 
22페이지, 누적220건 수집 최종2025.09.25 
23페이지, 누적230건 수집 최종2025.09.11 
24페이지, 누적240건 수집 최종2025.08.28 
25페이지, 누적250건 수집 최종2025.08.13 
2025.08.12, 1년 이전 날짜까지 데이터크롤링 완료
최종 250건 · 2025.08.13 ~ 2026.08.24


### 문항 3 네이버 뉴스 검색기 함수 만들기 (스크롤 포함) - 난이도 상

In [ ]:
# 키워드를 받아 네이버 뉴스 검색 결과를 수집하는 함수를 작성하시오.

# 대상: https://search.naver.com/search.naver?ssc=tab.news.all

# crawl_naver_news(keyword, page) — 아래로 스크롤해야 나오는 결과까지 수집, ※page 수 만큼 스크롤

# 추출 필드: 제목 / 언론사 / 링크 / 요약

# Selenium 사용 금지. 스크롤이 유발하는 요청을 Network에서 찾아 재현할 것

# 결과를 news_{keyword}.csv로 저장할 것

In [35]:
from urllib.parse import unquote

In [67]:
URL = 'https://s.search.naver.com/p/newssearch/3/api/tab/more'

param_str = '?abt=null&cluster_rank=78&de=&ds=&eid=&field=0&force_original=&is_dts=0&is_sug_officeid=0&mynews=0&news_office_checked=&nlu_query=&nqx_theme=%7B%22theme%22%3A%7B%22main%22%3A%7B%22name%22%3A%22health%22%2C%22source%22%3A%22NLU%22%2C%22score%22%3A%22100.000000%22%7D%7D%7D&nso=so%3Ar%2Cp%3Aall%2Ca%3Aall&nx_and_query=&nx_search_hlquery=&nx_search_query=&nx_sub_query=&office_category=&office_section_code=0&office_type=0&pd=0&photo=0&qdt=0&query=AI&query_original=&rev=0&service_area=&sm=tab_smr&sort=0&spq=0&ssc=tab.news.all&start=21'
params = [p.split('=') for p in param_str.split('&')]
params ={k: unquote(v) for k, v in params  }

keyword = 'AI'
start = 1 #11, 21, 31, 41, 51, 61

res = requests.get(URL, params={**params, 'query':keyword, 'start':start})
res.raise_for_status()
res.json()['collection'][0]['html'].find('만드는 사람만')

7481

In [68]:
soup = BeautifulSoup(res.json()['collection'][0]['html'], 'html.parser')

In [115]:
news = soup.select('.fds-news-item-list-tab > div')
news_crawl = []
for new in news:
    
    naver_news_link = new.select_one('a[href^="https://n.news.naver.com"]')
    if naver_news_link:
        link = naver_news_link.attrs['href']
    else:
        link = new.select_one('a[data-heatmap-target=".tit"]').attrs['href']
    news_crawl.append({
        '제목': new.select_one('.sds-comps-text-type-headline1').text.strip(),
        '언론사': new.select_one('.sds-comps-profile-info-title a span').text.strip(),
        '링크': link,
        '요약': new.select_one('.sds-comps-text-type-body1').text.strip(),
    })
news_crawl

[{'제목': '"AI, 만드는 사람만 책임지나"…정부, 이용자까지 AI 윤리원칙 세운다',
  '언론사': '뉴시스',
  '링크': 'https://n.news.naver.com/mnews/article/003/0014143861?sid=105',
  '요약': '생성형 인공지능(AI)이 일상과 산업 전반으로 빠르게 확산하면서 AI를 만드는 기업뿐 아니라 이를 사용하는 이용자의 책임도 중요해지고 있다. 이에 정부는 새 AI 윤리원칙을 통해 그동안 개발자나 제공자 중심으로 여겨졌던 AI 윤리 실천의 범위를 이용자로 넓힌다. 동시에 법적 구속력이 없는 자율규범으로...'},
 {'제목': "삼성SDS, 서울대 'AI 네이티브 캠퍼스' 구축한다",
  '언론사': '연합뉴스',
  '링크': 'https://n.news.naver.com/mnews/article/001/0016266002?sid=105',
  '요약': "이번 사업은 인공지능(AI) 연구와 교육에 필요한 고성능 네트워크 인프라를 구축해 서울대를 AI 시대에 맞춘 'AI 네이티브 캠퍼스'로 전환하는 프로젝트다. 서울대는 243개 동에서 약 5만명이 사용하는 캠퍼스 네트워크의 유·무선 품질을 개선한다. 또한 AI 연구용 대용량 데이터를 빠르게 전송할 수 있도록..."},
 {'제목': "엔비디아, 10조원 베팅해 '개방형 AI' 승부…중국 모델에 맞불",
  '언론사': '파이낸셜뉴스',
  '링크': 'https://n.news.naver.com/mnews/article/014/0005565225?sid=101',
  '요약': '인공지능(AI) 반도체 최강자 엔비디아가 중국의 첨단 AI 모델에 맞서 세계 최고 수준의 오픈웨이트(개방형 가중치) 모델 개발에 본격적으로 뛰어든다. AI 스타트업 풀사이드와 약 70억달러(약 9조7000억원) 규모의 기술 제휴·지분 투자를 단행하고 개발 인력까지 대거 확보하기로 했다. 22일(현지시간)...'},
 {'제목': "KT, 전사 IT 'AI 체질'로 바꾼다

In [126]:
### 최적화
URL = 'https://s.search.naver.com/p/newssearch/3/api/tab/more'

param_str = '?abt=null&cluster_rank=78&de=&ds=&eid=&field=0&force_original=&is_dts=0&is_sug_officeid=0&mynews=0&news_office_checked=&nlu_query=&nqx_theme=%7B%22theme%22%3A%7B%22main%22%3A%7B%22name%22%3A%22health%22%2C%22source%22%3A%22NLU%22%2C%22score%22%3A%22100.000000%22%7D%7D%7D&nso=so%3Ar%2Cp%3Aall%2Ca%3Aall&nx_and_query=&nx_search_hlquery=&nx_search_query=&nx_sub_query=&office_category=&office_section_code=0&office_type=0&pd=0&photo=0&qdt=0&query=AI&query_original=&rev=0&service_area=&sm=tab_smr&sort=0&spq=0&ssc=tab.news.all&start=21'
params = [p.split('=') for p in param_str.split('&')]
params ={k: unquote(v) for k, v in params  }

def naver_news_crawl(keyword, page):
    news_crawl = []
    for start in range(page):
        start = start * 10 + 1
        res = requests.get(URL, params={**params, 'query':keyword, 'start':start})
        res.raise_for_status()
        

        soup = BeautifulSoup(res.json()['collection'][0]['html'], 'html.parser')

        news = soup.select('.fds-news-item-list-tab > div')
    
        for new in news:
            
            naver_news_link = new.select_one('a[href^="https://n.news.naver.com"]')
            if naver_news_link:
                link = naver_news_link.attrs['href']
            else:
                link = new.select_one('a[data-heatmap-target=".tit"]').attrs['href']
            news_crawl.append({
                '제목': new.select_one('.sds-comps-text-type-headline1').text.strip(),
                '언론사': new.select_one('.sds-comps-profile-info-title a span').text.strip(),
                '링크': link,
                '요약': new.select_one('.sds-comps-text-type-body1').text.strip(),
            })
            time.sleep(0.7)
    return news_crawl

result = []
result = naver_news_crawl('AI', 10)
df = pd.DataFrame(result).to_csv(f'news_{keyword}.csv',index=False, encoding='utf-8-sig')
print(f'{len(result)}건 수집 완료')

100건 수집 완료
